# 밀링 머신 정비 상담 에이전트

작업자가 설비 상태를 말로 물으면, LLM이 학습된 고장 예측 모델(ML/DL)을 직접 호출해서 판정 근거와 조치를 답합니다.

- **LLM 에이전트**: LangChain `bind_tools` + OpenAI Tool Calling (5장)
- **데이터**: AI4I 2020 Predictive Maintenance Dataset (UCI, CC BY 4.0, 밀링 머신 10,000건, 고장 3.4%)
- **ML**: 로지스틱 회귀, 랜덤포레스트 (고장 여부 + 고장 유형 5종)
- **DL**: TensorFlow/Keras 심층 신경망 (Dense + Dropout + EarlyStopping + class_weight)
- **LLM**: LangChain LCEL 체인 (`ChatPromptTemplate | LLM | StrOutputParser`), OpenAI 또는 Ollama(gemma2)
- **환각 방지**: 판정은 코드가 정하고 LLM은 설명만, 출력 판정 자동 검증

| 파일 | 클래스 | 역할 |
|---|---|---|
| `src/data_loader.py` | `AI4IDataLoader` | 데이터 로드 · 파생 변수 · 분할 |
| `src/dl_model.py` | `KerasFailureClassifier` | Keras 심층 신경망 |
| `src/models.py` | `FailurePredictor` | ML/DL 학습 · 평가 · 예측 |
| `src/rule_checker.py` | `FailureRuleChecker` | 고장 기준값 비교 |
| `src/llm_reporter.py` | `MaintenanceReporter` | LangChain 정비 메모 생성 · 판정 검증 |
| `src/agent.py` | `MaintenanceAgent` | LLM 에이전트 · 도구 · 답변 검증 |

## 0. 환경 준비 (Colab)
프로젝트 zip을 올려서 압축을 풉니다. 로컬 실행 시에는 건너뛰세요.

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import files
    uploaded = files.upload()          # 프로젝트 zip 업로드
    zip_name = next(iter(uploaded))
    !unzip -o -q "$zip_name" -d /content/project
    %cd /content/project
    !pip install -q -r requirements.txt

## 1. 데이터 로드 · 파생 변수
고장 원리에 맞춰 온도차(열 방출), 소비 전력(전력 고장), 공구 부하(과부하)를 추가합니다.

In [ ]:
from src.data_loader import AI4IDataLoader

loader = AI4IDataLoader()
df = loader.load()
print(loader.summary())
df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["temp_diff", "power", "strain"]):
    df[df.machine_failure == 0][col].plot.hist(bins=50, alpha=.6, ax=ax, label="normal", density=True)
    df[df.machine_failure == 1][col].plot.hist(bins=50, alpha=.6, ax=ax, label="failure", density=True)
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

## 2. ML/DL 모델 학습 · 비교
고장이 3.4%뿐이라 모두 '정상'이라고 해도 정확도 96.6%가 나옵니다. 그래서 **F1 · 재현율 · ROC-AUC**로 비교합니다.

In [ ]:
import pandas as pd
from src.models import FailurePredictor

X_train, X_test, y_train, y_test, yt_train, yt_test = loader.split()
predictor = FailurePredictor().fit(X_train, y_train, yt_train)
metrics = predictor.evaluate(X_test, y_test, yt_test)

pd.DataFrame(metrics["binary"]).T.drop(columns="confusion_matrix")

### Keras 심층 신경망 학습 곡선

In [ ]:
hist = predictor.dl_history()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["loss"], label="train"); axes[0].plot(hist["val_loss"], label="validation")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist["auc"], label="train"); axes[1].plot(hist["val_auc"], label="validation")
axes[1].set_title("AUC"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()
print("학습 epoch 수:", len(hist["loss"]))

### 고장 유형별 성능 · 특성 중요도

In [ ]:
print("대표 모델:", metrics["best_model"])
pd.DataFrame(metrics["failure_types"]).T

In [ ]:
predictor.feature_importance().iloc[::-1].plot.barh(figsize=(7, 4), title="Feature importance")
plt.show()
predictor.save()

## 3. 단일 설비 진단 + 기준값 비교

In [ ]:
from src.rule_checker import FailureRuleChecker

K = 273.15
X_row = AI4IDataLoader.make_single("L", air_temp=25.9 + K, process_temp=36.4 + K,
                                   rpm=1350, torque=60.0, tool_wear=210)
sensor = X_row.iloc[0].to_dict()
pred = predictor.predict_one(X_row)
findings = FailureRuleChecker().check(sensor)

print(pred)
for f in findings:
    print("기준 초과" if f["triggered"] else "정상", "|", f["name"], "|", f["detail"])

## 4. LangChain 정비 메모 생성
- 판정(정상/주의/위험)은 코드가 정하고, LLM은 근거와 조치만 씁니다.
- LLM이 판정을 다르게 쓰면 `verify_verdict()`가 자동으로 바로잡습니다.
- `provider="ollama"`이면 로컬 gemma2(무료), `"openai"`이면 OpenAI API를 씁니다.

In [ ]:
import os, getpass
from IPython.display import Markdown
from src.llm_reporter import MaintenanceReporter

PROVIDER = "ollama"   # "ollama" 또는 "openai"
if PROVIDER == "openai" and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

reporter = MaintenanceReporter(provider=PROVIDER)
print("체인:", reporter.get_chain())
Markdown(reporter.generate(sensor, pred, findings))

## 5. LLM 정비 상담 에이전트 (핵심)
작업자의 질문을 LLM이 이해하고, 학습된 모델을 도구로 호출해서 답합니다.
- `diagnose`: 고장 확률 · 판정 · 기준 초과 항목 계산
- `suggest_adjustments`: 공구 교체 · 토크 낮추기 · 회전수 조정의 효과를 모델로 계산
- 답변 검증: 도구 결과에 없는 수치 감지, 가정한 입력값 자동 표시

Tool Calling이 필요해서 OpenAI 키가 있어야 합니다.

In [ ]:
from src.agent import MaintenanceAgent

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

agent = MaintenanceAgent(predictor, provider="openai")
history = []
for q in ["L등급 제품인데 토크 60Nm, 공구 210분 썼어. 계속 돌려도 돼?",
          "그럼 어떻게 하면 돼? 공구 교체 말고 다른 방법도 있어?",
          "토크만 40으로 낮추면?"]:
    answer, trace, history = agent.run(history, q)
    print("작업자:", q)
    print("호출한 도구:", [(t["tool"], t["args"]) for t in trace])
    print(answer)
    print("근거 없는 수치:", MaintenanceAgent.check_numbers(answer, trace) or "없음")
    print("-" * 60)

## 6. 시연
- 로컬: `streamlit run app.py`
- 배포: Streamlit Community Cloud (README 참고)